# 01. INGEST NY_TAXI DATA

In [2]:
# Download Green Taxi data from Jan 2025 to Jul 2025
import wget
wget.download('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-11.parquet')

100% [..........................................................................] 1164775 / 1164775

'green_tripdata_2025-11.parquet'

In [3]:
import numpy as np
import pandas as pd

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

In [5]:
import time
start_time = time.perf_counter()
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Elapsed time for sparkSession.builder: {elapsed_time:.4f} seconds")

Elapsed time for sparkSession.builder: 0.6992 seconds


In [8]:
import time

start_time = time.perf_counter()
#For reading data in Spark dataframe
df_green = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .parquet('green_tripdata_2025-11.parquet')
end_time = time.perf_counter()

elapsed_time = end_time - start_time
print(f"Elapsed time for spark.read: {elapsed_time:.4f} seconds")

#For reading data in Spark SQL
start_time = time.perf_counter()
df_green.registerTempTable('green_2025')
end_time = time.perf_counter()

elapsed_time = end_time - start_time
print(f"Elapsed time for register temptable: {elapsed_time:.4f} seconds")

Elapsed time for spark.read: 2.7455 seconds


C:\Users\HP\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\sql\classic\dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


Elapsed time for register temptable: 1.9357 seconds


In [9]:
#partitioning the dataframe for easier load
df_green.repartition(4)

DataFrame[VendorID: int, lpep_pickup_datetime: timestamp_ntz, lpep_dropoff_datetime: timestamp_ntz, store_and_fwd_flag: string, RatecodeID: bigint, PULocationID: int, DOLocationID: int, passenger_count: bigint, trip_distance: double, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, ehail_fee: double, improvement_surcharge: double, total_amount: double, payment_type: bigint, trip_type: bigint, congestion_surcharge: double, cbd_congestion_fee: double]

In [10]:
#use Pandas dataframe to make the display cleaner to see
import pandas as pd
pd.set_option('display.max_columns', None)
df_green.limit(5).toPandas()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,2,2025-11-01 00:34:48,2025-11-01 00:41:39,N,1,74,42,1,0.74,7.2,1.00,0.5,1.94,0.0,NaN,1.0,11.64,1,1,0.00,0.0
1,2,2025-11-01 00:18:52,2025-11-01 00:24:27,N,1,74,42,2,0.95,7.2,1.00,0.5,0.00,0.0,NaN,1.0,9.70,2,1,0.00,0.0
2,2,2025-11-01 01:03:14,2025-11-01 01:15:24,N,1,83,160,1,2.19,13.5,1.00,0.5,5.00,0.0,NaN,1.0,21.00,1,1,0.00,0.0
3,2,2025-11-01 00:10:57,2025-11-01 00:24:53,N,1,166,127,1,5.44,24.7,1.00,0.5,0.50,0.0,NaN,1.0,27.70,1,1,0.00,0.0
4,1,2025-11-01 00:03:48,2025-11-01 00:19:38,N,1,166,262,1,3.20,18.4,3.75,1.5,1.00,0.0,NaN,1.0,24.65,1,1,2.75,0.0


In [12]:
import wget
wget.download('https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv')

100% [..............................................................................] 12331 / 12331

'taxi_zone_lookup.csv'

In [13]:
#For reading zone data in Spark dataframe
df_zone = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv('taxi_zone_lookup.csv')

#For reading zone data in Spark SQL
df_zone.registerTempTable('taxi_zone')

C:\Users\HP\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\sql\classic\dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [14]:
df_zone.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

# 02. TRANSFORM TABLE USING SPARK SQL

In [15]:
#Use Case 1: Count the trip for each day
start_time = time.perf_counter()
daily_trip = spark.sql("""
SELECT to_date(lpep_pickup_datetime) as date, count(1) as trip_count from green_2025
group by to_date(lpep_pickup_datetime) 
order by to_date(lpep_pickup_datetime);
""").toPandas()
display(daily_trip)
end_time = time.perf_counter()

elapsed_time = end_time - start_time
print(f"Elapsed time for loading daily trip data: {elapsed_time:.4f} seconds")

,date,trip_count
0,2025-10-26,1
1,2025-10-27,1
2,2025-10-30,1
3,2025-10-31,5
4,2025-11-01,1465
5,2025-11-02,1295
6,2025-11-03,1642
7,2025-11-04,1470
8,2025-11-05,1804
9,2025-11-06,1862


Elapsed time for loading daily trip data: 3.3230 seconds


There are the data outside November 2025 trip, which is outlier. we want to check and remove the outlier data 

In [19]:
#check the outlier data
spark.sql ("""
SELECT to_date(lpep_pickup_datetime) as date, lpep_pickup_datetime, lpep_dropoff_datetime 
from green_2025
where to_date(lpep_pickup_datetime) < '2025-11-01' or to_date(lpep_pickup_datetime) > '2025-11-30'
order by date
""").toPandas()

,date,lpep_pickup_datetime,lpep_dropoff_datetime
0,2025-10-26,2025-10-26 20:23:16,2025-10-26 20:31:18
1,2025-10-27,2025-10-27 20:20:49,2025-10-27 20:47:57
2,2025-10-30,2025-10-30 22:11:19,2025-10-30 22:25:53
3,2025-10-31,2025-10-31 23:55:08,2025-11-01 00:25:30
4,2025-10-31,2025-10-31 19:35:25,2025-10-31 20:01:21
5,2025-10-31,2025-10-31 23:58:40,2025-11-01 00:26:34
6,2025-10-31,2025-10-31 19:44:41,2025-10-31 19:54:57
7,2025-10-31,2025-10-31 19:10:31,2025-10-31 19:31:14
8,2025-12-01,2025-12-01 08:05:06,2025-12-01 08:18:01
9,2025-12-01,2025-12-01 08:46:16,2025-12-01 08:59:14


In [24]:
#cleaned the data to only trip in november 2025
cleaned_green_2025 = spark.sql ("""
SELECT * from green_2025
where to_date(lpep_pickup_datetime) > '2025-10-31' and to_date(lpep_pickup_datetime) < '2025-12-01';
""")

cleaned_green_2025.registerTempTable('cleaned_green_2025')

C:\Users\HP\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\sql\classic\dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [25]:
daily_trip = spark.sql("""
SELECT to_date(lpep_pickup_datetime) as date, count(1) as trip_count from cleaned_green_2025
group by to_date(lpep_pickup_datetime) 
order by to_date(lpep_pickup_datetime);
""").toPandas()
display(daily_trip)

,date,trip_count
0,2025-11-01,1465
1,2025-11-02,1295
2,2025-11-03,1642
3,2025-11-04,1470
4,2025-11-05,1804
5,2025-11-06,1862
6,2025-11-07,1645
7,2025-11-08,1398
8,2025-11-09,1228
9,2025-11-10,1729


In [26]:
#Store Use Case 1 to csv for load to bigquery
daily_trip.to_csv("daily_trip.csv", index=False)

In [27]:
# Use Case 2: Identifying top 5 most frequent pickup zone
start_time = time.perf_counter()
most_pickup = spark.sql ("""
    SELECT dfz.LocationID, dfz.Zone, count(1) as trip_count
    FROM cleaned_green_2025 df
    JOIN taxi_zone dfz ON df.PULocationID = dfz.locationID
    GROUP BY dfz.locationID, dfz.Zone
    ORDER BY count(1) desc
""").limit(10).toPandas()
end_time = time.perf_counter()

elapsed_time = end_time - start_time
print(f"Elapsed time for loading daily trip data: {elapsed_time:.4f} seconds")
display(most_pickup)

Elapsed time for loading daily trip data: 2.1029 seconds


,LocationID,Zone,trip_count
0,74,East Harlem North,12041
1,75,East Harlem South,5836
2,166,Morningside Heights,2181
3,95,Forest Hills,2122
4,43,Central Park,1935
5,41,Central Harlem,1799
6,97,Fort Greene,1777
7,82,Elmhurst,1755
8,65,Downtown Brooklyn/MetroTech,1384
9,130,Jamaica,1380


In [28]:
#Store Use Case 2 to csv for load to bigquery
most_pickup.to_csv("most_pickup.csv", index=False)

In [39]:
# Use Case 3: Identifying top 5 routes
top_routes = spark.sql ("""
    SELECT 
    CONCAT(pu.Zone, ' --> ', do.Zone) AS hailing_route,
    COUNT(1) as trip_count
FROM 
    cleaned_green_2025 df 
    LEFT JOIN taxi_zone pu ON df.PULocationID = pu.LocationID
    LEFT JOIN taxi_zone do ON df.DOLocationID = do.LocationID
    GROUP BY hailing_route
    ORDER BY trip_count DESC
""").limit(10).toPandas()
end_time = time.perf_counter()

display(top_routes)

,hailing_route,trip_count
0,East Harlem North --> East Harlem South,1790
1,East Harlem North --> Upper East Side North,1414
2,East Harlem South --> East Harlem North,1014
3,East Harlem North --> Morningside Heights,879
4,East Harlem North --> Upper West Side North,767
5,East Harlem North --> Yorkville West,752
6,Forest Hills --> Forest Hills,719
7,East Harlem North --> Central Harlem,680
8,East Harlem North --> Central Harlem North,516
9,Elmhurst --> LaGuardia Airport,483


In [40]:
#Store Use Case 1 to csv for load to bigquery
top_routes.to_csv("top_routes.csv", index=False)

# 03. LOAD TRANSFROMED TABLE TO BIGQUERY

In [28]:
!pip install google.cloud

In [35]:
!pip install --upgrade google-cloud-bigquery

  Using cached google_cloud_bigquery-3.40.1-py3-none-any.whl.metadata (8.2 kB)
  Using cached google_cloud_core-2.5.0-py3-none-any.whl.metadata (3.1 kB)
  Using cached google_resumable_media-2.8.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached protobuf-6.33.5-cp310-abi3-win_amd64.whl.metadata (593 bytes)
  Using cached proto_plus-1.27.1-py3-none-any.whl.metadata (2.2 kB)
  Using cached grpcio_status-1.78.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached cryptography-46.0.5-cp311-abi3-win_amd64.whl.metadata (5.7 kB)
  Using cached rsa-4.9.1-py3-none-any.whl.metadata (5.6 kB)
  Using cached pyasn1-0.6.2-py3-none-any.whl.metadata (8.4 kB)
Using cached google_cloud_bigquery-3.40.1-py3-none-any.whl (262 kB)
Using cached google_cloud_core-2.5.0-py3-none-any.whl (29 kB)
Using cached google_resumable_media-2.8.0-py3-none-any.whl (81 kB)
   ---------------------------------------- 0.0/4.9 MB ? eta -:--:--
   ---- ------

In [34]:
import google

from google.cloud import bigquery
from google.oauth2 import service_account

In [35]:
credentials = service_account.Credentials.from_service_account_file(
    "zoomcamp-2026-486804-f2b3a82300fb.json"
)

client = bigquery.Client(credentials=credentials)

In [36]:
table_id = "zoomcamp-2026-486804.ny_taxi_spark.daily_trip"

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
)

with open("daily_trip.csv", "rb") as source_file:
    job = client.load_table_from_file(
        source_file,
        table_id,
        job_config=job_config
    )
job.result()

print("Daily Trip CSV successfully loaded to BigQuery.")

Daily Trip CSV successfully loaded to BigQuery.


In [37]:
table_id = "zoomcamp-2026-486804.ny_taxi_spark.most_pickup"

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
)

with open("most_pickup.csv", "rb") as source_file:
    job = client.load_table_from_file(
        source_file,
        table_id,
        job_config=job_config
    )
job.result()

print("Most Pickup CSV successfully loaded to BigQuery.")

Most Pickup CSV successfully loaded to BigQuery.


In [41]:
table_id = "zoomcamp-2026-486804.ny_taxi_spark.top_routes"

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
)

with open("top_routes.csv", "rb") as source_file:
    job = client.load_table_from_file(
        source_file,
        table_id,
        job_config=job_config
    )
job.result()

print("Top Routes CSV successfully loaded to BigQuery.")

Top Routes CSV successfully loaded to BigQuery.
